# Speculative sampling, checked by simulation

This notebook checks the speculative decoding results in notes.md §5.4. There
are no neural networks: the "language models" are explicit categorical
distributions over a 6-token vocabulary, because the claims are about the
sampling algorithm, not about any particular model.

The claim (Leviathan et al. 2023; Chen et al. 2023): a small draft model q
proposes tokens, the large target model p verifies them, and an accept/reject
rule keeps or replaces each proposal. The emitted tokens are distributed
exactly according to p. For one position:

1. sample x from q
2. accept x with probability min(1, p(x)/q(x))
3. on rejection, sample from the residual p'(x) ∝ max(0, p(x) - q(x))

The proof is in notes.md §5.4. This notebook checks, by sampling:

- Part 1: one position. The emitted distribution matches p, and the
  acceptance rate equals Σ min(p, q) = 1 - TV(p, q).
- Part 2: a context-dependent toy LM (a Markov chain) with the full k-token
  draft-and-verify loop. The distribution over whole sequences matches
  ordinary sampling from p.
- Part 3: expected tokens emitted per target pass, (1 - a^(k+1)) / (1 - a),
  compared against a simulated count, for drafts of varying quality.

`uv run jupyter execute ml/07_llm_stack/speculative_decoding_demo.ipynb`

## Setup and the one-position rule

`tv` is the total variation distance, half the L1 distance between two
distributions. `spec_step` performs one position of speculative sampling and
returns the emitted token and whether the draft was accepted. The residual is
normalized by its sum, which equals the rejection probability (notes.md §5.4).

In [1]:
import numpy as np

rng = np.random.default_rng(0)
V = 6  # vocabulary size


def tv(p, q):
    # total variation distance = half the L1 distance
    return 0.5 * np.abs(p - q).sum()


def spec_step(p, q, rng):
    # one position of speculative sampling; returns (token, accepted?)
    x = rng.choice(V, p=q)
    if rng.random() < min(1.0, p[x] / q[x]):          # accept w.p. min(1, p/q)
        return x, True
    resid = np.maximum(p - q, 0.0)                     # residual (p - q)^+
    resid /= resid.sum()                               # its sum = P(reject)
    return rng.choice(V, p=resid), False

## Part 1: one position, emitted distribution vs p

We draw a random target p and a different random draft q, run `spec_step`
100,000 times, and compare the empirical distribution of emitted tokens with
both. Expect the TV distance to p to be on the order of the sampling noise
(about 1/√n ≈ 0.003), the TV distance to q to be much larger, and the
empirical acceptance rate to match Σ min(p, q). The assert fails if the
distance to p exceeds five times the noise scale.

In [2]:
def part1(n=100_000):
    print("=" * 66)
    print("PART 1: single-position accept/reject, %d samples" % n)
    print("=" * 66)
    p = rng.dirichlet(np.ones(V) * 2)   # target
    q = rng.dirichlet(np.ones(V) * 2)   # a deliberately different draft

    counts = np.zeros(V)
    accepts = 0
    for _ in range(n):
        x, acc = spec_step(p, q, rng)
        counts[x] += 1
        accepts += acc
    emp = counts / n

    print(f"{'token':>6} {'target p':>10} {'draft q':>10} {'empirical':>10}")
    for i in range(V):
        print(f"{i:>6} {p[i]:>10.4f} {q[i]:>10.4f} {emp[i]:>10.4f}")
    print(f"\n  TV(empirical, target p) = {tv(emp, p):.4f}   "
          f"(sampling noise scale ~ {1 / np.sqrt(n):.4f})")
    print(f"  TV(empirical, draft  q) = {tv(emp, q):.4f}   "
          "(far from q: the output follows p, not the draft)")
    pred_acc = np.minimum(p, q).sum()
    print(f"  acceptance rate: empirical {accepts / n:.4f}  vs  "
          f"predicted sum(min(p,q)) = {pred_acc:.4f} = 1 - TV(p,q)")
    assert tv(emp, p) < 5 / np.sqrt(n), "emitted distribution != target!"


part1()

PART 1: single-position accept/reject, 100000 samples


 token   target p    draft q  empirical
     0     0.1607     0.0166     0.1600
     1     0.2310     0.0813     0.2292
     2     0.0934     0.1655     0.0933
     3     0.3487     0.3529     0.3519
     4     0.0800     0.2355     0.0814
     5     0.0862     0.1481     0.0843

  TV(empirical, target p) = 0.0047   (sampling noise scale ~ 0.0032)
  TV(empirical, draft  q) = 0.2912   (far from q: the output follows p, not the draft)
  acceptance rate: empirical 0.7084  vs  predicted sum(min(p,q)) = 0.7063 = 1 - TV(p,q)


## Part 2: the full k-token loop on a toy LM

The toy LM is a first-order Markov chain: the distribution of the next token
given the previous one is a row of a 6×6 transition matrix. Target and draft
are two different matrices. `spec_generate` follows the real algorithm: the
draft proposes k tokens one after another, the target checks them in order
(in a real system this is one parallel forward pass; here we read rows of its
matrix), the first rejection is replaced by a residual sample, and if all k
are accepted one extra token is sampled from the target. `ancestral_generate`
is ordinary token-by-token sampling from the target, for comparison. The cell
generates one short sequence each way as a smoke test.

In [3]:
def make_lm(concentration, rng):
    return rng.dirichlet(np.ones(V) * concentration, size=V)  # V x V, rows sum to 1


def spec_generate(P, Q, length, k, rng):
    # Generate `length` tokens with speculative decoding.
    # Returns (tokens, n_target_passes); one target pass checks up to k drafts.
    out = [0]  # fixed start token (dropped from the returned sequence)
    passes = 0
    while len(out) - 1 < length:
        drafts = []                                # draft proposes k tokens from Q
        ctx = out[-1]
        for _ in range(k):
            x = rng.choice(V, p=Q[ctx])
            drafts.append(x)
            ctx = x
        passes += 1                                # target checks all k in one pass
        ctx = out[-1]
        all_accepted = True
        for x in drafts:
            p, q = P[ctx], Q[ctx]
            if rng.random() < min(1.0, p[x] / q[x]):
                out.append(x)                      # accept
                ctx = x
            else:                                  # reject: residual sample, stop
                resid = np.maximum(p - q, 0.0)
                resid /= resid.sum()
                out.append(int(rng.choice(V, p=resid)))
                all_accepted = False
                break
            if len(out) - 1 == length:
                break
        # all k accepted: the target's next-token distribution was already
        # computed in the same pass, so sample one more token from it
        if all_accepted and len(out) - 1 < length:
            out.append(int(rng.choice(V, p=P[out[-1]])))
    return out[1:length + 1], passes


def ancestral_generate(P, length, rng):
    out = [0]
    for _ in range(length):
        out.append(int(rng.choice(V, p=P[out[-1]])))
    return out[1:]


P_demo = make_lm(2.0, rng)
Q_demo = 0.5 * P_demo + 0.5 * make_lm(2.0, rng)
toks, passes = spec_generate(P_demo, Q_demo, length=12, k=3, rng=rng)
print("speculative:", toks, f"({passes} target passes for {len(toks)} tokens)")
print("ancestral:  ", ancestral_generate(P_demo, 12, rng))

speculative: [4, 4, 2, 2, 4, 4, 5, 0, 4, 1, 1, 1] (4 target passes for 12 tokens)
ancestral:   [2, 4, 5, 5, 0, 4, 1, 0, 2, 4, 0, 5]


## Part 2 (continued): sequence distribution vs exact p

We sample 100,000 sequences of length 3 with speculative decoding (k = 3) and
100,000 with ancestral sampling, and compare both empirical distributions over
all 6³ = 216 sequences with the exact probabilities under the target chain.
Expect the two TV distances to be about equal, since both differ from the
exact distribution only by sampling noise, and more than 1 token per target
pass for speculative decoding. This cell takes about half a minute.

In [4]:
def part2(n=100_000, length=3, k=3):
    print("=" * 66)
    print(f"PART 2: Markov LM, sequences of length {length}, draft k={k}, {n} samples")
    print("=" * 66)
    P = make_lm(2.0, rng)                      # target LM
    Q = 0.5 * P + 0.5 * make_lm(2.0, rng)      # draft: partly agrees with P

    def seq_id(s):
        i = 0
        for t in s:
            i = i * V + t
        return i

    n_seqs = V ** length
    spec_counts = np.zeros(n_seqs)
    anc_counts = np.zeros(n_seqs)
    total_passes = total_tokens = 0
    for _ in range(n):
        s, passes = spec_generate(P, Q, length, k, rng)
        spec_counts[seq_id(s)] += 1
        total_passes += passes
        total_tokens += length
        anc_counts[seq_id(ancestral_generate(P, length, rng))] += 1
    spec_emp, anc_emp = spec_counts / n, anc_counts / n

    exact = np.zeros(n_seqs)                   # exact sequence probabilities under P
    for i in range(n_seqs):
        s, prob, prev = i, 1.0, 0
        digits = []
        for _ in range(length):
            digits.append(s % V)
            s //= V
        for t in reversed(digits):
            prob *= P[prev][t]
            prev = t
        exact[i] = prob

    print(f"  TV(speculative, exact target) = {tv(spec_emp, exact):.4f}")
    print(f"  TV(ancestral,   exact target) = {tv(anc_emp, exact):.4f}   "
          "(same noise level: speculative sampling is exact)")
    print(f"  tokens per target pass: {total_tokens / total_passes:.2f}  "
          "(vs 1.0 for plain autoregressive decoding)")
    assert tv(spec_emp, exact) < 2 * tv(anc_emp, exact) + 0.01


part2()

PART 2: Markov LM, sequences of length 3, draft k=3, 100000 samples


  TV(speculative, exact target) = 0.0146
  TV(ancestral,   exact target) = 0.0161   (same noise level: speculative sampling is exact)
  tokens per target pass: 2.57  (vs 1.0 for plain autoregressive decoding)


## Part 3: acceptance rate and tokens per pass vs draft quality

Here p and q do not depend on context, so every position is accepted with the
same probability a = Σ min(p, q) and the formula from notes.md §5.4 holds
exactly: expected tokens per target pass = 1 + a + ... + a^k =
(1 - a^(k+1)) / (1 - a). The draft is a blend `q = w*p + (1-w)*bad`, where
`bad` always proposes the token that p considers least likely; w = 1 is a
perfect draft and w = 0 a very poor one. For each w the cell
computes the formula and simulates 20,000 target passes (k = 4 drafts each,
counting accepted drafts plus the one residual or extra token). Expect the
two columns to agree within about 0.02, k + 1 = 5 tokens per pass for the
perfect draft, and close to 1 token per pass (never below 1) for the poor one.

In [5]:
def simulate_tokens_per_pass(p, q, k, n, rng):
    # each pass: k drafts from q; emitted = (accepted drafts before the first
    # rejection) + 1 (the residual sample, or the extra token if all accepted)
    xs = rng.choice(V, size=(n, k), p=q)
    accept = rng.random((n, k)) < np.minimum(1.0, p[xs] / q[xs])
    n_accepted = np.where(accept.all(axis=1), k, np.argmin(accept, axis=1))
    return (n_accepted + 1).mean()


def part3(k=4, n=20_000):
    print("=" * 66)
    print(f"PART 3: acceptance and tokens per pass vs draft agreement (k={k})")
    print("=" * 66)
    p = rng.dirichlet(np.ones(V) * 2)
    bad = np.eye(V)[np.argmin(p)]              # always proposes p's least likely token
    print(f"{'mix w':>6} {'TV(p,q)':>9} {'accept a':>9} "
          f"{'formula (1-a^(k+1))/(1-a)':>27} {'simulated':>10}")
    for w in [1.0, 0.9, 0.7, 0.5, 0.3, 0.0]:
        q = w * p + (1 - w) * bad              # draft = target blended with a bad draft
        a = np.minimum(p, q).sum()             # acceptance = 1 - TV(p, q)
        formula = (1 - a ** (k + 1)) / (1 - a) if a < 1 else k + 1
        sim = simulate_tokens_per_pass(p, q, k, n, rng)
        print(f"{w:>6.1f} {tv(p, q):>9.4f} {a:>9.4f} {formula:>27.2f} {sim:>10.2f}")
        assert abs(sim - formula) < 0.05, "simulation disagrees with the formula"
    print("\n  A perfect draft (w=1) gives k+1 tokens per target pass. A poor draft")
    print("  still gives at least 1 (the residual sample), so tokens per pass never")
    print("  falls below plain decoding; the loss is only the draft model's compute.")


part3()
print("\nAll checks passed: speculative sampling reproduces the target distribution,")
print("and the tokens-per-pass formula matches simulation.")

PART 3: acceptance and tokens per pass vs draft agreement (k=4)
 mix w   TV(p,q)  accept a   formula (1-a^(k+1))/(1-a)  simulated
   1.0    0.0000    1.0000                        5.00       5.00
   0.9    0.0934    0.9066                        4.15       4.15
   0.7    0.2801    0.7199                        2.88       2.88
   0.5    0.4669    0.5331                        2.05       2.05
   0.3    0.6536    0.3464                        1.52       1.52
   0.0    0.9337    0.0663                        1.07       1.07

  A perfect draft (w=1) gives k+1 tokens per target pass. A poor draft
  still gives at least 1 (the residual sample), so tokens per pass never
  falls below plain decoding; the loss is only the draft model's compute.

All checks passed: speculative sampling reproduces the target distribution,
and the tokens-per-pass formula matches simulation.
